In [0]:
import dlt
import pyspark.sql.functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType, DateType
)
 
# ============================================================
# BRONZE LAYER - Raw Ingestion (Auto Loader + Rescued Columns)
# ============================================================
 
@dlt.table(
    name="sales_bronze",
    comment="Bronze: raw incremental sales files via Auto Loader (with rescued data)"
)
def sales_bronze():
    return (
        spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "csv")
            .option("header", "true")
            .option("cloudFiles.inferColumnTypes", "true")
            .option("cloudFiles.schemaLocation", "abfss://raw@nilaystore.dfs.core.windows.net/checkpoints/sales_schema")
            .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
            .option("rescuedDataColumn", "_rescued_data")
            .load("abfss://raw@nilaystore.dfs.core.windows.net/sales")
    )
 
@dlt.table(
    name="products_bronze",
    comment="Bronze: raw product metadata (with rescued data)"
)
def products_bronze():
    return (
        spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "csv")
            .option("header", "true")
            .option("cloudFiles.inferColumnTypes", "true")
            .option("cloudFiles.schemaLocation", "abfss://raw@nilaystore.dfs.core.windows.net/checkpoints/products_schema")
            .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
            .option("rescuedDataColumn", "_rescued_data")
            .load("abfss://raw@nilaystore.dfs.core.windows.net/products")
    )
 
# Expose rescued rows for inspection
@dlt.table(name="sales_bronze_rescued", comment="Rescued rows from sales ingestion")
def sales_bronze_rescued():
    return dlt.read("sales_bronze").filter(F.col("_rescued_data").isNotNull())
 
@dlt.table(name="products_bronze_rescued", comment="Rescued rows from products ingestion")
def products_bronze_rescued():
    return dlt.read("products_bronze").filter(F.col("_rescued_data").isNotNull())
 
# ============================================================
# SILVER LAYER - Cleaned Data + Expectations + Quarantine
# ============================================================
 
@dlt.table(
    name="nilay2_catalog.silver.sales_silver",
    comment="Silver: cleaned sales (types, null filters, dedupe)",
    table_properties={"delta.enableChangeDataFeed": "true"}
)
@dlt.expect("Valid sales", "sales_amount >= 0")
@dlt.expect("Valid quantity", "quantity > 0")
def sales_silver():
    df = dlt.read("sales_bronze")
    df2 = (
        df
        .filter(F.col("sale_id").isNotNull() & F.col("product_id").isNotNull())
        .withColumn("quantity", F.col("quantity").cast("int"))
        .withColumn("sales_amount", F.col("sales_amount").cast("double"))
        .withColumn("sale_date", F.to_date(F.col("sale_date"), "yyyy-MM-dd"))
        .withColumn(
            "sales_amount_calculated",
            (F.col("quantity") * (F.col("sales_amount") /
              F.when(F.col("quantity") == 0, F.lit(1)).otherwise(F.col("quantity"))))
        )
    )
    return df2.dropDuplicates(["sale_id"])
 
# Quarantine table for expectation failures
@dlt.table(
    name="sales_silver_quarantine",
    comment="Rows failing sales expectations (invalid sales_amount or quantity)"
)
def sales_silver_quarantine():
    df = dlt.read("sales_bronze")
    return df.filter(
        (F.col("sales_amount").cast("double") < 0) |
        (F.col("quantity").cast("int") <= 0)
    )
 
@dlt.table(
    name="nilay2_catalog.silver.products_silver",
    comment="Silver: cleaned products; handles new discount_rate column"
)
def products_silver():
    df = dlt.read("products_bronze")
    df2 = (
        df
        .withColumn("price", F.col("price").cast("double"))
        .withColumn("discount_rate", F.coalesce(F.col("discount_rate").cast("double"), F.lit(0.0)))
    )
    return df2.dropDuplicates(["product_id"])
 
# ============================================================
# GOLD LAYER - Business Aggregates
# ============================================================
 
@dlt.table(
    name="nilay2_catalog.gold.daily_revenue_region",
    comment="Gold: daily revenue aggregated by region"
)
def gold_daily_revenue_region():
    df = dlt.read("nilay2_catalog.silver.sales_silver")
    return (
        df.groupBy("sale_date", "region")
          .agg(
              F.round(F.sum("sales_amount").cast("double"), 2).alias("daily_revenue"),
              F.count("sale_id").alias("num_transactions")
          )
    )
 
@dlt.table(
    name="nilay2_catalog.gold.product_performance",
    comment="Gold: total sales per product (with product metadata)"
)
def gold_product_performance():
    sales = dlt.read("nilay2_catalog.silver.sales_silver")
    products = dlt.read("nilay2_catalog.silver.products_silver")
    joined = sales.join(products, on="product_id", how="left")
    return (
        joined.groupBy("product_id", "product_name", "category")
              .agg(
                  F.round(F.sum("sales_amount").cast("double"), 2).alias("total_sales_amount"),
                  F.round(F.avg("discount_rate"), 4).alias("avg_discount_rate"),
                  F.count("sale_id").alias("total_transactions")
              )
    )
 
# ============================================================
# MONITORING TABLE - Expectation Summary
# ============================================================
 
@dlt.table(
    name="sales_silver_expectation_summary",
    comment="Summary of sales_silver expectation outcomes by batch"
)
def sales_silver_expectation_summary():
    df = dlt.read("sales_bronze")
    return (
        df.groupBy(F.current_date().alias("as_of_date"))
          .agg(
              F.count("*").alias("rows_seen"),
              F.sum((F.col("sales_amount").cast("double") < 0).cast("int")).alias("neg_sales_rows"),
              F.sum((F.col("quantity").cast("int") <= 0).cast("int")).alias("bad_quantity_rows"),
              F.sum(F.col("_rescued_data").isNotNull().cast("int")).alias("rescued_rows")
          )
    )
 
 